# Khai b?o th? vi?n

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import keras as kr
from keras.models import Sequential
from keras.layers import Dense
from sklearn import metrics


#B??c 1: ??c file csv

In [ ]:
data = pd.read_csv('/content/Churn_Modelling.csv')
data = data.drop(['CustomerId', 'Surname', 'RowNumber'], axis = 1)
data

#B??c 2: Chuy?n ??i d? li?u d?ng kh?ng th? t? sang d?ng onehot encoding

In [ ]:
X = data.iloc[:,0:10]
y = data.iloc[:,10]

In [ ]:
X = pd.get_dummies(X)
X

#Chia t?p d? li?u train v? test theo t? l? 8 : 2

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

#B??c 4: Chu?n h?a b?ng Standard Scaler

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)


#B??c 5: Build m?ng Neural Network v?i ki?n tr?c: Input - 6 neuron/ReLU - 6 neuron/ReLU - 1 neuron/sigmoid

In [ ]:
model = Sequential()
n_features = X_train.shape[1]
# model.add(Dense(6, activation='relu'))
model.add(Dense(6, input_dim=n_features , activation='relu'))
model.add(Dense(6, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

#B??c 6: Hu?n luy?n v?i thu?t to?n Adam v? h?m ?? l?i Binary Cross Entropy

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])


In [ ]:
model.fit(X_train, y_train, epochs=50, batch_size=10)

#B??c 7: Test v? ??nh gi? ?? ch?nh x?c c?a m? h?nh

In [ ]:
_, accuracy = model.evaluate(X_train, y_train)
print('Accuracy: %.2f' % (accuracy*100))

#Th? nghi?m xem v?i kh?ch h?ng c? th?ng tin sau th? c? n?n exit hay ko?

In [ ]:
new_prediction = model.predict(scaler.transform(np.array([[600, 40 , 3, 60000, 2, 1, 1, 50000, 1, 0, 0, 0, 1]])))

new_prediction = (new_prediction > 0.5 )
print(new_prediction)

#Kfold


In [ ]:
kfold = KFold(n_splits = 5, random_state= None)

In [ ]:
def get_model():

  model = Sequential()
  n_features = X_train.shape[1]
  # model.add(Dense(6, activation='relu'))
  model.add(Dense(6, input_dim=n_features , activation='relu'))
  model.add(Dense(6, activation='relu'))
  model.add(Dense(1, activation='sigmoid'))
  model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

  return model

In [ ]:
  X = np.concatenate((X_train, X_test), axis=0)
  y = np.concatenate((y_train, y_test), axis=0)

In [ ]:
# ??nh ngh?a K-Fold CV
kfold = KFold(n_splits=5, shuffle=True)

loss_list =[]
accuracy_list = []
# K-fold Cross Validation model evaluation
fold_idx = 1

for train_ids, val_ids in kfold.split(X, y):

  model = get_model()

  print("B?t ??u train Fold ", fold_idx)

  # Train model
  model.fit(X[train_ids], y[train_ids],
              batch_size=10,
              epochs= 50,
              verbose=1)

  # Test v? in k?t qu?
  scores = model.evaluate(X[val_ids], y[val_ids], verbose=0)
  print("?? train xong Fold ", fold_idx)

  # Th?m th?ng tin accuracy v? loss v?o list
  accuracy_list.append(scores[1] * 100)
  loss_list.append(scores[0])

  # Sang Fold ti?p theo
  fold_idx = fold_idx + 1

In [ ]:
print('* Chi ti?t c?c fold')
for i in range(0, len(accuracy_list)):
  print(f'> Fold {i+1} - Loss: {loss_list[i]} - Accuracy: {accuracy_list[i]}%')

print('* ??nh gi? t?ng th? c?c folds:')
print(f'> Accuracy: {np.mean(accuracy_list)} (?? l?ch +- {np.std(accuracy_list)})')
print(f'> Loss: {np.mean(loss_list)}')